# Vorhersage-Workflow mittels importierter Daten-Pipeline

Dieses Notebook nutzt die ausgelagerte `non_labled_data_pipeline`, um die Testdaten aufzubereiten. Der Fokus liegt hier auf dem Laden des Modells und der Erstellung der finalen Vorhersagen.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path
from catboost import CatBoostClassifier

# Importieren der Verarbeitungsfunktion aus Ihrer Pipeline-Datei
from pipeline.non_labled_data_pipeline import process_unlabeled_data

## Schritt 1: Laden des trainierten Modells und des Preprocessors

Wir laden die zuvor gespeicherten Artefakte: das trainierte CatBoost-Modell und den gefitteten `preprocessor`.

In [ ]:
# --- Pfade definieren und neueste Dateien finden ---
weights_dir = Path('./weights')

# Finde die neueste Modelldatei (.cbm)
model_files = sorted([f for f in weights_dir.iterdir() if f.suffix == '.cbm'], reverse=True)
if not model_files:
    raise FileNotFoundError("Keine .cbm Modelldatei im Ordner ./weights gefunden.")
latest_model_path = model_files[0]

# Finde die neueste Preprocessor-Datei (.joblib)
preprocessor_files = sorted([f for f in weights_dir.iterdir() if f.suffix == '.joblib'], reverse=True)
if not preprocessor_files:
    raise FileNotFoundError("Keine .joblib Preprocessor-Datei im Ordner ./weights gefunden.")
latest_preprocessor_path = preprocessor_files[0]


# --- Modell und Preprocessor laden ---
print(f"Lade Modell von: {latest_model_path}")
loaded_model = CatBoostClassifier()
loaded_model.load_model(latest_model_path)

print(f"Lade Preprocessor von: {latest_preprocessor_path}")
loaded_preprocessor = joblib.load(latest_preprocessor_path)

print("\n✅ Modell und Preprocessor erfolgreich geladen.")

Lade Modell von: weights\final_catboost_regressor_20250701_004513.cbm
Lade Preprocessor von: weights\preprocessor_regression_20250701_004513.joblib

✅ Modell und Preprocessor erfolgreich geladen.


## Schritt 2: Testdaten mit der Pipeline verarbeiten

Jetzt rufen wir die Funktion `process_unlabeled_data` aus unserer Pipeline auf. Diese Funktion übernimmt das Laden, Aggregieren und Transformieren der rohen Testdaten. Sie gibt die verarbeiteten Features für das Modell und die Originaldaten (für die IDs) zurück.

In [ ]:
# Die gesamte Datenverarbeitung wird durch einen einzigen Funktionsaufruf erledigt
X_test_processed, X_test_full = process_unlabeled_data(loaded_preprocessor)

print("\nKopf der verarbeiteten Daten:")
print(X_test_processed.head())

Lade rohe Testdaten...
Aggregiere Testdaten...


C:\Users\lol--\AppData\Local\Temp\ipykernel_23520\1079550483.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aggregated_lines = lines_enriched.groupby('transaction_id').apply(


✅ Testdaten-Aggregation abgeschlossen.
Form der aggregierten Testdaten: (811905, 15)


## Schritt 3: Vorhersagen erstellen

Mit den aufbereiteten Daten werden nun die Vorhersagen unter Anwendung des optimierten Schwellenwerts von **0.47** getroffen.

In [5]:
print("Erstelle Vorhersagen...")
# Wahrscheinlichkeiten für die Klasse 'FRAUD' (Klasse 1) erhalten
y_scores = loaded_model.predict_proba(X_test_processed)[:, 1]

# Den optimalen Threshold aus dem Training anwenden
optimal_threshold = 0.47
y_pred_optimized = (y_scores >= optimal_threshold).astype(int)

print(f"✅ Vorhersagen mit Threshold {optimal_threshold} abgeschlossen.")

Transformiere Testdaten mit dem Preprocessor...


c:\Users\lol--\miniconda3\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


KeyboardInterrupt: 

## Schritt 4: Vorhersage-Datei erstellen und speichern

Die finalen Vorhersagen werden zusammen mit den zugehörigen Transaktions-IDs in einer CSV-Datei gespeichert.

In [6]:
# Erstelle das DataFrame für die Abgabe
# Wir verwenden X_test_full, um die ursprünglichen IDs zu erhalten
submission_df = pd.DataFrame({
    'id': X_test_full['id'],
    'prediction': y_pred_optimized
})

# Speichere die Ergebnisse als CSV-Datei
output_file = 'catboost_predictions_via_pipeline.csv'
submission_df.to_csv(output_file, index=False)

print(f"✅ Vorhersage-Datei '{output_file}' erfolgreich gespeichert.")
print("\nErste 5 Zeilen der Vorhersage-Datei:")
print(submission_df.head())

# Zähle die Anzahl der Vorhersagen pro Klasse
print("\nVerteilung der Vorhersagen:")
print(submission_df['prediction'].value_counts())

NameError: name 'y_pred_optimized' is not defined